In [ ]:
# INITIAL SETTINGS

!pip install -U kaleido
# basic python libraries
import os
import pickle
import warnings
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
# regressors
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
!pip install catboost
from catboost import CatBoostRegressor
# hyperparameter tuning
!pip install optuna
import optuna
from optuna.samplers import TPESampler
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate
# PyTorch for NN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
# SHAP for explainability
import shap

warnings.filterwarnings('ignore')
seed = 2006
np.random.seed(seed)
torch.manual_seed(seed)

# paths for saving of results
base_path = '/content/drive/MyDrive/Concrete Compressive Strength Prediction'
cleaned_data = os.path.join(base_path, '01_Dataset', 'Processed', 'df_clean.csv')
processed_dir = os.path.join(base_path, '01_Dataset', 'Processed')
models_baseline = os.path.join(base_path, '04_Models', 'baseline')
models_tuned  = os.path.join(base_path, '04_Models','tuned')
models_dl      = os.path.join(base_path, '04_Models','ANN')
metrics_dir    = os.path.join(base_path, '04_Results','metrics')
optuna_dir     = os.path.join(base_path, '04_Results','optuna_studies')
plots_perf     = os.path.join(base_path, '04_Results','plots','model_performance')
plots_shap     = os.path.join(base_path, '04_Results','plots','shap')
# create if not exist
for d in [processed_dir, models_baseline, models_tuned, models_dl,
          metrics_dir, optuna_dir, plots_perf, plots_shap]:
    os.makedirs(d, exist_ok=True)

In [ ]:
import shutil
import os
from google.colab import drive

# Clean up the local folder preventing the mount
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')

# Now mount Google Drive
drive.mount('/content/drive')

# Re-create folders now that the mount is active
for d in [processed_dir, models_baseline, models_tuned, models_dl,
          metrics_dir, optuna_dir, plots_perf, plots_shap]:
    os.makedirs(d, exist_ok=True)

# Read the data
df = pd.read_csv(cleaned_data, index_col=0)
df.shape

In [ ]:
df.duplicated().value_counts()

In [ ]:
df.isna().sum()

In [ ]:
df.sample(5)

In [ ]:
# Reset index so 'Cement' is included as a feature
df_clean = df.reset_index()

# Strip potential leading/trailing whitespaces from column names
df_clean.columns = df_clean.columns.str.strip()

# Separate Features & Labels
target_col = 'Concrete compressive strength(MPa, megapascals)'
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print(f"Columns in df_clean: {list(df_clean.columns)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
X.columns

In [ ]:
y.name

In [ ]:
# Train/Test Spli -> 80 / 20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# StandardScaler
scaler = StandardScaler()
# create a copy scaling not required for tree & boosting algos
X_train_scal = X_train.copy()
X_test_scal = X_test.copy()
X_train_scaled = scaler.fit_transform(X_train_scal)
X_test_scaled = scaler.transform(X_test_scal)
# keep as DF also for future reference
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
# save for future reference as train.csv & test.csv
pd.concat([X_train,y_train],axis=1).to_csv(os.path.join(processed_dir,'train.csv'),index=False)
pd.concat([X_test,y_test],axis=1).to_csv(os.path.join(processed_dir,'test.csv'),index=False)
pd.concat([X_train_scaled,y_test],axis=1).to_csv(os.path.join(processed_dir,'train_scaled.csv'),index=False)
pd.concat([X_test_scaled,y_test],axis=1).to_csv(os.path.join(processed_dir,'test_scaled.csv'),index=False)

In [ ]:
X_train_scaled.mean() # scaling effective , all approx 0

In [ ]:
X_train_scaled.std() # scaling effective , all approx 1

In [ ]:
# save the scaler
with open(os.path.join(processed_dir,'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
def evaluate_model(model, X_train,y_train,X_test,y_test,cv=5):
  y_pred = model.predict(X_test)
  r2 = r2_score(y_test,y_pred)
  rmse = np.sqrt(mean_squared_error(y_test,y_pred))
  mae = mean_absolute_error(y_test,y_pred)
  cv_scores = cross_val_score(model,X_train,y_train,cv=cv,scoring='r2')
  return {
      'R2': r2,
      'RMSE': rmse,
      'MAE': mae,
      'CV_R2_mean': cv_scores.mean(),
      'CV_R2_std': cv_scores.std()
  }

# save the metric values
def save_metrics(metrics_dict,filename,append=True):
  df = pd.DataFrame([metrics_dict])
  mode = 'a' if append and os.path.exists(filename) else 'w'
  header = not os.path.exists(filename) or mode == 'w'
  df.to_csv(filename,mode=mode,header=header,index=False)

Baseline Models : i.e Models with default hyperparameters

In [ ]:
# Models requiring scaled data i.e Gradient Based & distance based algos
scaled_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=seed),              # L2 regularization
    'Lasso': Lasso(random_state=seed, max_iter=5000),   # L1 regularization
    'KNN': KNeighborsRegressor(),
    'SVR': SVR(verbose=True)
}

# Tree & boosting algos, no need for scaling
unscaled_models = {
    'DecisionTree': DecisionTreeRegressor(random_state=seed),
    'RandomForest': RandomForestRegressor(random_state=seed, verbose=1),
    'GradientBoosting': GradientBoostingRegressor(random_state=seed, verbose=1),
    'XGBoost': xgb.XGBRegressor(random_state=seed, verbosity=1),
    'LightGBM': lgb.LGBMRegressor(random_state=seed, verbose=-1),
    'CatBoost': CatBoostRegressor(random_state=seed, verbose=100)
}
baseline_results = []

# 1. Scaled Models
for name, model in scaled_models.items():
  print(f"-"*100)
  print(f"\nTraining {name} Started")
  start_time = time.perf_counter()
  model.fit(X_train_scaled,y_train)
  end_time = time.perf_counter()
  elapsed_time = end_time - start_time
  # helper function called
  metrics = evaluate_model(model,X_train_scaled,y_train,X_test_scaled,y_test)
  metrics['Model'] = name
  metrics['Type'] = 'Baseline'
  metrics['Scaled'] = True
  metrics['Training Time (s)'] = round(elapsed_time,4)
  # helper function called below
  baseline_results.append(metrics)
  # save the results
  with open(os.path.join(models_baseline, f'{name}.pkl'), 'wb') as f:
    pickle.dump(model,f)
  save_metrics(metrics, os.path.join(metrics_dir, 'baseline_results.csv'))
  print(f"\nTraining {name} Finished")
  print(f"\nTraining Time: {elapsed_time}")
  print(f"-"*100)
for name, model in unscaled_models.items():
  print(f"-"*100)
  print(f"\nTraining {name} Started")
  start_time = time.perf_counter()
  model.fit(X_train, y_train)
  end_time = time.perf_counter()
  elapsed_time = end_time - start_time
  metrics = evaluate_model(model, X_train, y_train, X_test, y_test)
  metrics['Model'] = name
  metrics['Type'] = 'Baseline'
  metrics['Scaled'] = False
  metrics['Training Time (s)'] = round(elapsed_time,4)
  baseline_results.append(metrics)
  with open(os.path.join(models_baseline, f'{name}.pkl'), 'wb') as f:
      pickle.dump(model, f)
  save_metrics(metrics, os.path.join(metrics_dir, 'baseline_results.csv'))
  print(f"\nTraining {name} Finished")
  print(f"\nTraining Time: {elapsed_time}")
  print(f"-"*100)

**Hyper parameter tuning using OPTUNA**

In [ ]:
# 80/20 split used for boosting algos to enable early stopping
X_train_boost, X_val_boost, y_train_boost, y_val_boost = train_test_split(
    X_train, y_train, test_size=0.2, random_state=seed
)
print(f"Boosting ALgos set:")
print(f"Training set: {X_train_boost.shape[0]} samples")
print(f"Validation set: {X_val_boost.shape[0]} samples")

In [ ]:
X_train_boost.head(5) # confirmed, using unscaled sets

**Hyperparameter Tuning**

In [ ]:
# OPTUNA study run funct & loop

n_trials = 75

def run_optuna_tuning(model_name, X_train_cv, y_train_cv, X_val_cv, y_val_cv,
                      X_train_full, y_train_full, X_test_eval, y_test_eval,
                      n_trials=n_trials):
    print(f"\n{'-'*100}")
    print(f"Tuning {model_name} with {n_trials} trials")
    print(f"{'-'*100}")

    def objective(trial):
        # hyperparameter search space
        if model_name == 'RandomForest':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
            }
            model = RandomForestRegressor(random_state=seed, n_jobs=-1, **params)
            return cross_val_score(model, X_train_cv, y_train_cv, cv=5, scoring='r2').mean()

        elif model_name == 'GradientBoosting':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0)
            }
            model = GradientBoostingRegressor(random_state=seed, **params)
            return cross_val_score(model, X_train_cv, y_train_cv, cv=5, scoring='r2').mean()

        elif model_name == 'XGBoost':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
            }
            model = xgb.XGBRegressor(random_state=seed, n_jobs=-1, verbosity=0, early_stopping_rounds=15, **params)
            model.fit(X_train_cv, y_train_cv, eval_set=[(X_val_cv, y_val_cv)], verbose=False)
            return r2_score(y_val_cv, model.predict(X_val_cv))

        elif model_name == 'LightGBM':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 10, 150),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0)
            }
            model = lgb.LGBMRegressor(random_state=seed, n_jobs=-1, verbose=-1, **params)
            model.fit(X_train_cv, y_train_cv, eval_set=[(X_val_cv, y_val_cv)], callbacks=[lgb.early_stopping(15, verbose=False)])
            return r2_score(y_val_cv, model.predict(X_val_cv))

        elif model_name == 'CatBoost':
            params = {
                'iterations': trial.suggest_int('iterations', 50, 300),
                'depth': trial.suggest_int('depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
            }
            model = CatBoostRegressor(random_state=seed, verbose=False, early_stopping_rounds=15, **params)
            model.fit(X_train_cv, y_train_cv, eval_set=[(X_val_cv, y_val_cv)], verbose=False)
            return r2_score(y_val_cv, model.predict(X_val_cv))

        else:
            raise ValueError(f"Model not identified")

    # run OPTUNA study
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # Save study object
    study_path = os.path.join(optuna_dir, f'{model_name}_study.pkl')
    with open(study_path, 'wb') as f:
        pickle.dump(study, f)

    # Generate and save Optuna perf plots
    try:
        fig1 = plot_optimization_history(study)
        fig1.write_image(os.path.join(plots_perf, f'{model_name}_optim_history.png'))
        fig2 = plot_param_importances(study)
        fig2.write_image(os.path.join(plots_perf, f'{model_name}_param_importance.png'))
        fig3 = plot_parallel_coordinate(study)
        fig3.write_image(os.path.join(plots_perf, f'{model_name}_parallel_coord.png'))
    except Exception as e:
        print(f"Plot generation failed for {model_name}: {e}")

    # Retrain with best params on full X_train
    best_params = study.best_params
    print(f"Best params for {model_name}: {best_params}")

    if model_name == 'RandomForest':
        final_model = RandomForestRegressor(random_state=seed, n_jobs=-1, **best_params)
    elif model_name == 'GradientBoosting':
        final_model = GradientBoostingRegressor(random_state=seed, **best_params)
    elif model_name == 'XGBoost':
        final_model = xgb.XGBRegressor(random_state=seed, n_jobs=-1, verbosity=0, **best_params)
    elif model_name == 'LightGBM':
        final_model = lgb.LGBMRegressor(random_state=seed, n_jobs=-1, verbose=-1, **best_params)
    elif model_name == 'CatBoost':
        final_model = CatBoostRegressor(random_state=seed, verbose=False, **best_params)

    final_model.fit(X_train_full, y_train_full)

    # Evaluate on test data
    metrics = evaluate_model(final_model, X_train_cv, y_train_cv, X_test_eval, y_test_eval)
    metrics['Model'] = model_name
    metrics['Type'] = 'Tuned'
    metrics['Scaled'] = False
    metrics['Best_Params'] = str(best_params)

    # Save tuned model
    model_path = os.path.join(models_tuned, f'{model_name}_tuned.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(final_model, f)

    # Save metrics to tuned_results.csv
    save_metrics(metrics, os.path.join(metrics_dir, 'tuned_results.csv'))

    return best_params, study, metrics

# run tuning for all 5
top5 = ['CatBoost', 'LightGBM', 'XGBoost', 'RandomForest', 'GradientBoosting']
tuned_metrics = {}

for model_name in top5:
    # For boosted models, using validation split
    if model_name in ['XGBoost', 'LightGBM', 'CatBoost']:
        X_tr = X_train_boost
        y_tr = y_train_boost
        X_val = X_val_boost
        y_val = y_val_boost
    else:
        # For RF and GB, using full X_train
        X_tr = X_train
        y_tr = y_train
        X_val = None
        y_val = None

    # pass arguments to the func
    best_params, study, metrics = run_optuna_tuning(
        model_name=model_name,
        X_train_cv=X_tr,
        y_train_cv=y_tr,
        X_val_cv=X_val,
        y_val_cv=y_val,
        X_train_full=X_train,
        y_train_full=y_train,
        X_test_eval=X_test,
        y_test_eval=y_test,
        n_trials=n_trials
    )
    tuned_metrics[model_name] = metrics
    print(f"\n{model_name} Best CV/Val R2 {study.best_value:.4f}")
    print(f"{model_name} Test R2 {metrics['R2']:.4f}")

print("\nTuning completed for all 5 candidates.")

In [ ]:
# plot utility didnt worked during optuna search
# regenerating using othe library
import plotly.io as pio
import os
pio.renderers.default = 'colab'

# the top 5
top_candidates = ['CatBoost', 'LightGBM', 'XGBoost', 'RandomForest', 'GradientBoosting']

# Ensure the output directory exists
os.makedirs(plots_perf, exist_ok=True)

for model_name in top_candidates:
    study_path = os.path.join(optuna_dir, f'{model_name}_study.pkl')

    if not os.path.exists(study_path):
        print(f"Study file not found for {model_name}, skipping.")
        continue

    # Load the study object
    with open(study_path, 'rb') as f:
        study = pickle.load(f)

    print(f"\nGenerating diagnostic plots for {model_name}")
    print(f"Best validation R2 from tuning: {study.best_value:.4f}")
# Generate the three figures
    fig1 = plot_optimization_history(study)
    fig2 = plot_param_importances(study)
    fig3 = plot_parallel_coordinate(study)

    # Display inline in the notebook
    print("Optimization history:")
    fig1.show()
    print("Parameter importance:")
    fig2.show()
    print("Parallel coordinate plot:")
    fig3.show()

    # Save to disk
    base_name = model_name
    png_suffix = '.png'
    html_suffix = '.html'

    try:
        # Try saving as PNG
        fig1.write_image(os.path.join(plots_perf, f'{base_name}_optim_history{png_suffix}'))
        fig2.write_image(os.path.join(plots_perf, f'{base_name}_param_importance{png_suffix}'))
        fig3.write_image(os.path.join(plots_perf, f'{base_name}_parallel_coord{png_suffix}'))
        print(f"PNG files saved for {model_name}")
    except Exception as e:
       # Fallback: save as interactive HTML
        print(f"Could not save PNG (kaleido missing or error). Saving HTML instead for {model_name}.")
        fig1.write_html(os.path.join(plots_perf, f'{base_name}_optim_history{html_suffix}'))
        fig2.write_html(os.path.join(plots_perf, f'{base_name}_param_importance{html_suffix}'))
        fig3.write_html(os.path.join(plots_perf, f'{base_name}_parallel_coord{html_suffix}'))
        print(f"HTML files saved for {model_name}")

print("\nAll diagnostic plots have been generated and saved.")

In [ ]:
# model comparison & final selection

# Load baseline and tuned results
baseline_path = os.path.join(metrics_dir, 'baseline_results.csv')
tuned_path = os.path.join(metrics_dir, 'tuned_results.csv')

baseline_df = pd.read_csv(baseline_path)
tuned_df = pd.read_csv(tuned_path)

# remove duplicate entries
#baseline_df = baseline_df.drop_duplicates(subset=['Model', 'Type'], keep='first')
#tuned_df = tuned_df.drop_duplicates(subset=['Model', 'Type'], keep='first')

# ensure numeric cols are float
numeric_cols = ['R2', 'RMSE', 'MAE', 'CV_R2_mean', 'CV_R2_std']
for df in [baseline_df, tuned_df]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# combine & sort
all_results = pd.concat([baseline_df, tuned_df], ignore_index=True)
all_results = all_results.dropna(subset=['R2'])  # remove rows with invalid R²
all_results_sorted = all_results.sort_values('R2', ascending=False)

# add generalization gap
all_results_sorted['Gap'] = all_results_sorted['CV_R2_mean'] - all_results_sorted['R2']
# Save the full comparison table
final_path = os.path.join(metrics_dir, 'final_comparison.csv')
all_results_sorted.to_csv(final_path, index=False)

print("\nTop 10 models (R²):")
print(all_results_sorted[['Model', 'Type', 'R2', 'RMSE', 'CV_R2_mean', 'Gap']].head(10).round(4))

# select the best model
# 1) Filter to models with R² > 0.92 and Gap < 0.02 (if any)
candidates = all_results_sorted[(all_results_sorted['R2'] > 0.92) & (all_results_sorted['Gap'] < 0.02)]

if not candidates.empty:
    # Among candidates, pick the one with highest R² (or lowest RMSE)
    best_model = candidates.sort_values('R2', ascending=False).iloc[0]
else:
    # Fallback: just take the highest R²
    best_model = all_results_sorted.iloc[0]

best_model_name = best_model['Model']
best_model_type = best_model['Type']
best_model_r2 = best_model['R2']
best_model_gap = best_model['Gap']

print("\n" + "="*60)
print("Final Model Selection")
print("="*60)
print(f"Selected model: {best_model_name} ({best_model_type})")
print(f"Test R²: {best_model_r2:.4f}")
print(f"CV/Val R²: {best_model['CV_R2_mean']:.4f}")
print(f"Generalization Gap: {best_model_gap:.4f}")
print("="*60)

# save best model infor for SHAP
best_info = {
    'Model': best_model_name,
    'Type': best_model_type,
    'Scaled': best_model.get('Scaled', False),  # default False if missing
    'R2': best_model['R2'],
    'RMSE': best_model['RMSE'],
    'MAE': best_model.get('MAE', None)
}
with open(os.path.join(metrics_dir, 'best_model_info.pkl'), 'wb') as f:
    pickle.dump(best_info, f)

print(f"\nBest model info saved to {os.path.join(metrics_dir, 'best_model_info.pkl')}")

In [ ]:
print(f"Generalization Gap: {best_model_gap:.4f}")
print("="*60)

# save best model info for SHAP
best_info = {
    'Model': best_model_name,
    'Type': best_model_type,
    'Scaled': best_model.get('Scaled', False),  # default False if missing
    'R2': best_model['R2'],
    'RMSE': best_model['RMSE'],
    'MAE': best_model.get('MAE', None)
}
with open(os.path.join(metrics_dir, 'best_model_info.pkl'), 'wb') as f:
    pickle.dump(best_info, f)

print(f"\nBest model info saved to {os.path.join(metrics_dir, 'best_model_info.pkl')}")

# Determine the correct model directory and file name dynamically
if best_model_type == 'Baseline':
    model_file_path = os.path.join(models_baseline, f'{best_model_name}.pkl')
else:
    model_file_path = os.path.join(models_tuned, f'{best_model_name}_tuned.pkl')

with open(model_file_path, 'rb') as f:
    model = pickle.load(f)

# prepare data
use_scaled = best_info['Scaled']
X_shap = X_test_scaled if use_scaled else X_test
feature_names = X_shap.columns.tolist()

# create TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)

print("SHAP values computed.")

# beeSwarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(os.path.join(plots_shap, 'global_summary_beeswarm.png'), dpi=150, bbox_inches='tight')
plt.plot()
print("Global summary plot saved.")

# dependence plots for top2 Features
feature_importance = np.abs(shap_values).mean(axis=0)
top_feature_indices = np.argsort(feature_importance)[::-1][:2]
top_feature_names = [feature_names[i] for i in top_feature_indices]
for idx, feat_name in zip(top_feature_indices, top_feature_names):
    plt.figure(figsize=(8, 6))
    shap.dependence_plot(idx, shap_values, X_shap, feature_names=feature_names, show=False)
    plt.title(f'SHAP Dependence Plot: {feat_name}')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_shap, f'dependence_{feat_name.replace("/", "_")}.png'), dpi=150)
    plt.plot()
    print(f"Dependence plot saved for {feat_name}")

In [ ]:
import os
import zipfile
from google.colab import files

# Path to store the zipped models
zip_filename = '/content/all_pkl_models.zip'

# Create a ZipFile object
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Walk through the base models directory to find all .pkl files
    models_root = os.path.join(base_path, '04_Models')
    for root, dirs, files_list in os.walk(models_root):
        for file in files_list:
            if file.endswith('.pkl'):
                file_path = os.path.join(root, file)
                # RelPath ensures structured folders in the zip archive
                arcname = os.path.relpath(file_path, models_root)
                zipf.write(file_path, arcname=arcname)
                print(f"Added: {arcname}")

print(f"\nZip archive created successfully: {zip_filename}")

# Trigger browser download of the zip file
files.download(zip_filename)

In [ ]:
import os
import zipfile
from google.colab import files

# Define the zip path for the datasets
datasets_zip = '/content/processed_datasets.zip'

# Create the zip archive
with zipfile.ZipFile(datasets_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(processed_dir):
        for file in files_list:
            # Add only dataset CSV and scaler files, skip any directories
            if file.endswith('.csv') or file == 'scaler.pkl':
                file_path = os.path.join(root, file)
                arcname = os.path.basename(file_path)
                zipf.write(file_path, arcname=arcname)
                print(f"Added: {arcname}")

print(f"\nZip archive created successfully: {datasets_zip}")

# Trigger browser download of the datasets zip
files.download(datasets_zip)

In [ ]:
import pandas as pd
import os

# Path to the final comparison file generated in previous steps
final_path = os.path.join(metrics_dir, 'final_comparison.csv')

if os.path.exists(final_path):
    # Load full comparison results
    results_df = pd.read_csv(final_path)

    # Select and rename columns to match the requested format
    # The dataset has: Model, Type, R2, RMSE, Gap
    col_mapping = {
        'Model': 'Model',
        'Type': 'Type',
        'R2': 'R²',
        'RMSE': 'RMSE (MPa)',
        'Gap': 'Gap'
    }

    # Filter to existing requested columns in the DataFrame
    columns_to_show = [col for col in col_mapping.keys() if col in results_df.columns]
    display_df = results_df[columns_to_show].rename(columns=col_mapping)

    print("Results")
    display(display_df.head(5).style.hide(axis='index'))
else:
    print(f"File not found at {final_path}. Make sure the comparison cells have been run.")